# iML1515 Evaluation

The goal of this notebook is to firstly evaluate iML1515, and see what transport reactions it facilitates, and what genes it has identified.\
The next step is to see what transporter genes the pipeline identify, and how the reactions these genes facilitate,differ and correspond with iML1515.

In [1]:
import json
import csv

In order to only retain what iML1515 has deems transport related reactions, one needs to look into the subsystems of the reactions. I have set the transport reactions to be defined by some simple keywords: By simply containing "transport" or "Extracellular exchange" in the subsystem name.

In [5]:
with open("iML1515.json", "r") as f:
    model = json.load(f)

def is_transport_subsystem(subsystem):
    return "transport" in subsystem.lower() or "extracellular exchange" in subsystem.lower()


# Create mappings for gene IDs to names, metabolite IDs to names, and a set of all genes in the model
gene_map = {g["id"]: g.get("name", "Unknown") for g in model["genes"]}
gene_ids_in_model = {g["id"] for g in model["genes"]}
metabolite_map = {m["id"]: m["name"] for m in model["metabolites"]}


# Store transport reactions and gene-tracking
transport_reactions = {}
all_genes = set()
genes_in_model = set()

# Iterate over all rxs in the model
for r in model["reactions"]:
    if "subsystem" in r and is_transport_subsystem(r["subsystem"]):
        metabolites = r.get("metabolites", {})

        # Extract gene IDs
        gene_ids = r.get("gene_reaction_rule", "").split()
        gene_ids_clean = [g for g in gene_ids if g.strip("()") not in ["and", "or"]]
        genes = [gene_map.get(g.strip("()"), g.strip("()")) for g in gene_ids_clean if g.strip("()")]

        # Track all genes associated with the rx
        all_genes.update(genes)

        # ID genes that are actually present in the model
        for g in gene_ids_clean:
            if g.strip("()") in gene_ids_in_model:
                genes_in_model.add(gene_map.get(g.strip("()"), "Unknown"))

        # Separate reactants and products for metabolite IDs
        reactants = [f"{-coef} {met}" for met, coef in metabolites.items() if coef < 0]
        products = [f"{coef} {met}" for met, coef in metabolites.items() if coef > 0]
        reaction_str = " + ".join(reactants) + " → " + " + ".join(products)

        # And metabolite names
        reactant_names = [f"{-coef} {metabolite_map.get(met, met)}" for met, coef in metabolites.items() if coef < 0]
        product_names = [f"{coef} {metabolite_map.get(met, met)}" for met, coef in metabolites.items() if coef > 0]
        reaction_str_names = " + ".join(reactant_names) + " → " + " + ".join(product_names)

        # Store details on rx
        transport_reactions[r["id"]] = {
            "name": r["name"],
            "subsystem": r.get("subsystem", "Unknown"),
            "reaction": reaction_str,
            "reaction_names": reaction_str_names,
            "genes": genes if genes else ["Unknown"]
        }

# Only include transport genes that are part of the model
genes_in_model_filtered = {gene for gene in all_genes if gene in genes_in_model}

print("Findings on transporters in iML1515\n")
print(f"Found {len(transport_reactions)} transport reactions.")
print(f"These transport reactions are coded for by {len(all_genes)} different genes.")
print(f"In iML1515, {len(genes_in_model_filtered)} of these genes are present.\n")
for rid, data in transport_reactions.items():
    print(f"{rid}: {data['name']} (Subsystem: {data['subsystem']})")
    print(f"\t{data['reaction']}")
    print(f"\t{data['reaction_names']}")
    print(f"\tGenes: {', '.join(data['genes'])}\n")

def save_tsv(fname):
    with open(fname, "w", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerow(["Reaction ID", "Name", "Subsystem", "Reaction (IDs)", "Reaction (Names)", "Genes"])
        
        for rid, data in transport_reactions.items():
            writer.writerow([rid, data["name"], data["subsystem"], data["reaction"], data["reaction_names"], ", ".join(data["genes"])])

save_tsv("iML1515_transport_rxs.tsv")

Findings on transporters in iML1515

Found 1148 transport reactions.
These transport reactions are coded for by 406 different genes.
In iML1515, 406 of these genes are present.

EX_pi_e: Phosphate exchange (Subsystem: Extracellular exchange)
	1.0 pi_e → 
	1.0 Phosphate → 
	Genes: Unknown

EX_co2_e: CO2 exchange (Subsystem: Extracellular exchange)
	1.0 co2_e → 
	1.0 CO2 CO2 → 
	Genes: Unknown

EX_met__L_e: L-Methionine exchange (Subsystem: Extracellular exchange)
	1.0 met__L_e → 
	1.0 L-Methionine → 
	Genes: Unknown

EX_metsox_S__L_e: L-Methionine S-oxide exchange (Subsystem: Extracellular exchange)
	1.0 metsox_S__L_e → 
	1.0 L-Methionine Sulfoxide → 
	Genes: Unknown

EX_acgam_e: N-Acetyl-D-glucosamine exchange (Subsystem: Extracellular exchange)
	1.0 acgam_e → 
	1.0 N-Acetyl-D-glucosamine → 
	Genes: Unknown

EX_cellb_e: Cellobiose exchange (Subsystem: Extracellular exchange)
	1.0 cellb_e → 
	1.0 Cellobiose → 
	Genes: Unknown

EX_crn_e: L-Carnitine exchange (Subsystem: Extracellular exc